# 졸음운전 Multi-task 학습 — 1 Model = 3 Tasks

하나의 checkpoint 안에 **Shared Backbone + Eye Head + Yawn Head + Head-Pose Head**를 넣습니다.

```text
MobileNetV2 / ResNet18 / MobileViT-XXS
                  ↓
            Shared Backbone
        ┌─────────┼─────────┐
        ↓                  ↓                  ↓
      Eye                  Yawn                Pose
   OPEN/CLOSED           NO/YAWN           Pitch/Yaw/Roll
```

최종 생성 파일은 총 3개입니다.

```text
mobilenetv2_multitask_best.pth
resnet18_multitask_best.pth
mobilevit_xxs_multitask_best.pth
```

데이터셋마다 보유 label이 다르므로 **partial-label multi-task learning**을 사용합니다. MRL batch에서는 Eye loss만, Yawn batch에서는 Yawn loss만, 300W-LP batch에서는 Pose loss만 계산하지만 세 task가 동일한 backbone을 업데이트합니다.

> MRL은 eye crop이고 Yawn/300W-LP는 face 이미지이므로 실시간에서는 **하나의 모델 인스턴스**에 `Eye ROI`와 `Face ROI`를 넣는 `forward_multitask()`을 사용합니다. 모델/checkpoint는 하나입니다.


In [1]:
!pip install -q timm kagglehub scikit-learn scipy pandas matplotlib

In [2]:
import gc, random, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from scipy.io import loadmat

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.transforms import InterpolationMode

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

import timm
import kagglehub

print('PyTorch:', torch.__version__)
print('timm   :', timm.__version__)
print('CUDA   :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU    :', torch.cuda.get_device_name(0))


PyTorch: 2.11.0+cu128
timm   : 1.0.28
CUDA   : True
GPU    : Tesla T4


In [3]:
# 2. 설정
SEED = 42
INPUT_SIZE = 256
EPOCHS = 20
BATCH_SIZE = 32
LR = 1e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 2
EARLY_STOP_PATIENCE = 5

MODELS_TO_RUN = [
    'mobilenetv2',
    'resnet18',
    'mobilevit_xxs',
]

MODEL_MAP = {
    'mobilenetv2': 'mobilenetv2_100',
    'resnet18': 'resnet18',
    'mobilevit_xxs': 'mobilevit_xxs.cvnets_in1k',
}

# Pose degree를 normalize해서 classification loss와 scale 차이를 줄임
POSE_SCALE = 90.0

# 빠른 코드 검증이 필요하면 숫자 지정, 최종 학습은 None 권장
HEAD_MAX_TRAIN_SAMPLES = None

# 각 epoch에서 세 task가 같은 batch 수를 사용
# None -> 세 train loader 중 가장 작은 batch 수 사용
STEPS_PER_TASK_PER_EPOCH = None

LAMBDA_EYE = 1.0
LAMBDA_YAWN = 1.0
LAMBDA_POSE = 1.0

RESULT_DIR = Path('/content/drowsiness_multitask_results')
RESULT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if torch.cuda.is_available():
        torch.backends.cudnn.benchmark = True

set_seed(SEED)
print('Device:', DEVICE)


Device: cuda


In [4]:
# 3. Dataset 다운로드
roots = {
    'eye': Path(kagglehub.dataset_download('tauilabdelilah/mrl-eye-dataset')),
    'yawn': Path(kagglehub.dataset_download('davidvazquezcic/yawn-dataset')),
    'head': Path(kagglehub.dataset_download('kameo4189/aflw2000-300wlp')),
}
for k,v in roots.items():
    print(k, '->', v)


100%|██████████| 330M/330M [00:03<00:00, 103MB/s] 

Extracting files...


100%|██████████| 16.9M/16.9M [00:00<00:00, 180MB/s]

Extracting files...


100%|██████████| 3.65G/3.65G [00:48<00:00, 80.8MB/s]

Extracting files...


eye -> /root/.cache/kagglehub/datasets/tauilabdelilah/mrl-eye-dataset/versions/6
yawn -> /root/.cache/kagglehub/datasets/davidvazquezcic/yawn-dataset/versions/1
head -> /root/.cache/kagglehub/datasets/kameo4189/aflw2000-300wlp/versions/29


In [5]:
# 4. Dataset index 생성
IMAGE_EXTS = {'.jpg','.jpeg','.png','.bmp','.pgm','.tif','.tiff'}

def norm(s):
    return str(s).lower().replace('-', '_').replace(' ', '_')

def rel_parts(path, root):
    path, root = Path(path), Path(root)
    try:
        p = path.relative_to(root)
    except ValueError:
        p = path
    return [norm(x) for x in p.parts]

def rel_text(path, root):
    return '/'.join(rel_parts(path, root))

# ---------- Eye: 0 OPEN / 1 CLOSED ----------
def eye_label(path, root):
    text = rel_text(path, root)
    if any(x in text for x in ['closed','close_eye','close_eyes','closed_eye','closed_eyes']):
        return 1
    if any(x in text for x in ['open','open_eye','open_eyes']):
        return 0
    return None

def source_split(path, root):
    parts = rel_parts(path, root)
    if any(x in {'test','testing'} for x in parts): return 'test'
    if any(x in {'val','valid','validation'} for x in parts): return 'val'
    if any(x in {'train','training'} for x in parts): return 'train'
    return 'unspecified'

def build_eye_df(root):
    rec=[]
    for p in Path(root).rglob('*'):
        if not p.is_file() or p.suffix.lower() not in IMAGE_EXTS: continue
        y=eye_label(p, root)
        if y is None: continue
        rec.append({'path':str(p),'label':y,'source_split':source_split(p,root)})
    df=pd.DataFrame(rec)
    if df.empty: raise RuntimeError('Eye OPEN/CLOSED label을 찾지 못했습니다.')
    return df

# ---------- Yawn: 0 NO_YAWN / 1 YAWN ----------
def yawn_label(path, root):
    text = rel_text(path, root)
    if any(x in text for x in ['no_yawn','noyawn','not_yawn','non_yawn','no_yawning']):
        return 0
    if 'yawn' in text:
        return 1
    return None

def build_yawn_df(root):
    rec=[]
    for p in Path(root).rglob('*'):
        if not p.is_file() or p.suffix.lower() not in IMAGE_EXTS: continue
        y=yawn_label(p, root)
        if y is None: continue
        rec.append({'path':str(p),'label':y})
    df=pd.DataFrame(rec)
    if df.empty: raise RuntimeError('YAWN/NO_YAWN label을 찾지 못했습니다.')
    return df

# ---------- Head Pose ----------
def read_pose(mat_path):
    m=loadmat(mat_path)
    if 'Pose_Para' not in m: raise KeyError('Pose_Para missing')
    pose=np.asarray(m['Pose_Para']).reshape(-1)
    if len(pose)<3: raise ValueError('Invalid Pose_Para')
    return tuple(float(x) for x in np.rad2deg(pose[:3].astype(np.float64)))

def is_aflw2000(path, root):
    # Kaggle root 이름 자체에 aflw2000이 있으므로 상대경로만 검사
    return any('aflw2000' in x or 'aflw2000_3d' in x for x in rel_parts(path,root))

def build_head_df(root):
    train_rec, test_rec, errors = [], [], []
    imgs=[p for p in Path(root).rglob('*') if p.is_file() and p.suffix.lower() in {'.jpg','.jpeg','.png'}]
    pairs=0
    for p in imgs:
        mat=p.with_suffix('.mat')
        if not mat.exists(): continue
        pairs += 1
        try:
            pitch,yaw,roll=read_pose(mat)
        except Exception as e:
            errors.append((str(p),str(e))); continue
        r={'path':str(p),'pitch':pitch,'yaw':yaw,'roll':roll}
        (test_rec if is_aflw2000(p,root) else train_rec).append(r)
    print('Head image files:',len(imgs),' image-mat pairs:',pairs)
    return pd.DataFrame(train_rec),pd.DataFrame(test_rec),errors

eye_all_df=build_eye_df(roots['eye'])
yawn_all_df=build_yawn_df(roots['yawn'])
head_all_df,head_test_df,head_errors=build_head_df(roots['head'])

print(); print('Eye:',len(eye_all_df))
print(eye_all_df.groupby(['source_split','label']).size())
print(); print('Yawn:',len(yawn_all_df), yawn_all_df['label'].value_counts().sort_index().to_dict())
print(); print('300W-LP:',len(head_all_df),' AFLW2000:',len(head_test_df),' errors:',len(head_errors))
if len(head_all_df)==0 or len(head_test_df)==0:
    raise RuntimeError('Head dataset 분리에 실패했습니다. 상대경로 예시를 확인하세요.')
print('300W-LP example:',head_all_df.iloc[0]['path'])
print('AFLW2000 example:',head_test_df.iloc[0]['path'])

Head image files: 124459  image-mat pairs: 124450

Eye: 84898
source_split  label
test          0         1657
              1         1566
train         0        41295
              1        40380
dtype: int64

Yawn: 5119 {0: 2591, 1: 2528}

300W-LP: 122450  AFLW2000: 2000  errors: 0
300W-LP example: /root/.cache/kagglehub/datasets/kameo4189/aflw2000-300wlp/versions/29/300W-LP/300W_LP/LFPW/LFPW_image_train_0493_4.jpg
AFLW2000 example: /root/.cache/kagglehub/datasets/kameo4189/aflw2000-300wlp/versions/29/AFLW2000-3D/AFLW2000/image01677.jpg


In [6]:
# 5. Train / Val / Test split
# Eye: 기존 train/test가 있으면 test 유지
cnt=eye_all_df['source_split'].value_counts()
if cnt.get('train',0)>0 and cnt.get('test',0)>0:
    base=eye_all_df[eye_all_df.source_split=='train'].copy()
    eye_test_df=eye_all_df[eye_all_df.source_split=='test'].reset_index(drop=True)
    eye_train_df,eye_val_df=train_test_split(base,test_size=0.12,random_state=SEED,stratify=base.label)
else:
    tv,eye_test_df=train_test_split(eye_all_df,test_size=0.15,random_state=SEED,stratify=eye_all_df.label)
    eye_train_df,eye_val_df=train_test_split(tv,test_size=0.15/0.85,random_state=SEED,stratify=tv.label)

eye_train_df=eye_train_df.reset_index(drop=True); eye_val_df=eye_val_df.reset_index(drop=True); eye_test_df=eye_test_df.reset_index(drop=True)

# Yawn
tv,yawn_test_df=train_test_split(yawn_all_df,test_size=0.15,random_state=SEED,stratify=yawn_all_df.label)
yawn_train_df,yawn_val_df=train_test_split(tv,test_size=0.15/0.85,random_state=SEED,stratify=tv.label)
yawn_train_df=yawn_train_df.reset_index(drop=True); yawn_val_df=yawn_val_df.reset_index(drop=True); yawn_test_df=yawn_test_df.reset_index(drop=True)

# Head Pose: angle 이상치 제거 후 300W-LP train/val, AFLW2000 test
ANGLE_LIMIT=99.0
def filter_pose(df):
    m=df.pitch.abs().le(ANGLE_LIMIT)&df.yaw.abs().le(ANGLE_LIMIT)&df.roll.abs().le(ANGLE_LIMIT)
    return df[m].reset_index(drop=True)
head_all_df=filter_pose(head_all_df); head_test_df=filter_pose(head_test_df)
if HEAD_MAX_TRAIN_SAMPLES is not None and len(head_all_df)>HEAD_MAX_TRAIN_SAMPLES:
    head_all_df=head_all_df.sample(HEAD_MAX_TRAIN_SAMPLES,random_state=SEED).reset_index(drop=True)
head_train_df,head_val_df=train_test_split(head_all_df,test_size=0.10,random_state=SEED)
head_train_df=head_train_df.reset_index(drop=True); head_val_df=head_val_df.reset_index(drop=True)

print('Eye :',len(eye_train_df),len(eye_val_df),len(eye_test_df))
print('Yawn:',len(yawn_train_df),len(yawn_val_df),len(yawn_test_df))
print('Head:',len(head_train_df),len(head_val_df),len(head_test_df))


Eye : 71874 9801 3223
Yawn: 3583 768 768
Head: 110173 12242 1969


In [7]:
# 6. Transform / Dataset / DataLoader
MEAN=(0.485,0.456,0.406); STD=(0.229,0.224,0.225)
cls_train_tf=transforms.Compose([
    transforms.RandomResizedCrop(INPUT_SIZE,scale=(0.82,1.0),interpolation=InterpolationMode.BICUBIC),
    transforms.RandomHorizontalFlip(0.5), transforms.RandomRotation(6),
    transforms.ColorJitter(brightness=0.15,contrast=0.15,saturation=0.10),
    transforms.ToTensor(),transforms.Normalize(MEAN,STD)])
pose_train_tf=transforms.Compose([
    transforms.RandomResizedCrop(INPUT_SIZE,scale=(0.90,1.0),interpolation=InterpolationMode.BICUBIC),
    transforms.ColorJitter(brightness=0.15,contrast=0.15,saturation=0.10),
    transforms.ToTensor(),transforms.Normalize(MEAN,STD)])
eval_tf=transforms.Compose([
    transforms.Resize(int(INPUT_SIZE/0.875),interpolation=InterpolationMode.BICUBIC),
    transforms.CenterCrop(INPUT_SIZE),transforms.ToTensor(),transforms.Normalize(MEAN,STD)])

class ClsDS(Dataset):
    def __init__(self,df,tf): self.df=df.reset_index(drop=True); self.tf=tf
    def __len__(self): return len(self.df)
    def __getitem__(self,i):
        r=self.df.iloc[i]; im=self.tf(Image.open(r.path).convert('RGB'))
        return im,int(r.label)

class PoseDS(Dataset):
    def __init__(self,df,tf): self.df=df.reset_index(drop=True); self.tf=tf
    def __len__(self): return len(self.df)
    def __getitem__(self,i):
        r=self.df.iloc[i]; im=self.tf(Image.open(r.path).convert('RGB'))
        y=torch.tensor([r.pitch,r.yaw,r.roll],dtype=torch.float32)/POSE_SCALE
        return im,y

def dl(ds,shuffle):
    return DataLoader(ds,batch_size=BATCH_SIZE,shuffle=shuffle,num_workers=NUM_WORKERS,pin_memory=DEVICE.type=='cuda')

loaders={
 'eye':{'train':dl(ClsDS(eye_train_df,cls_train_tf),True),'val':dl(ClsDS(eye_val_df,eval_tf),False),'test':dl(ClsDS(eye_test_df,eval_tf),False)},
 'yawn':{'train':dl(ClsDS(yawn_train_df,cls_train_tf),True),'val':dl(ClsDS(yawn_val_df,eval_tf),False),'test':dl(ClsDS(yawn_test_df,eval_tf),False)},
 'pose':{'train':dl(PoseDS(head_train_df,pose_train_tf),True),'val':dl(PoseDS(head_val_df,eval_tf),False),'test':dl(PoseDS(head_test_df,eval_tf),False)},
}
for t in loaders:
    print(t,{s:len(loaders[t][s].dataset) for s in loaders[t]},'batches',{s:len(loaders[t][s]) for s in loaders[t]})


eye {'train': 71874, 'val': 9801, 'test': 3223} batches {'train': 2247, 'val': 307, 'test': 101}
yawn {'train': 3583, 'val': 768, 'test': 768} batches {'train': 112, 'val': 24, 'test': 24}
pose {'train': 110173, 'val': 12242, 'test': 1969} batches {'train': 3443, 'val': 383, 'test': 62}


In [8]:
# 7. 하나의 Shared Backbone + 3 Heads
class MultiTaskDrowsinessModel(nn.Module):
    def __init__(self,model_key,pretrained=True,dropout=0.2):
        super().__init__(); self.model_key=model_key
        name=MODEL_MAP[model_key]
        try:
            self.backbone=timm.create_model(name,pretrained=pretrained,num_classes=0,global_pool='avg')
        except Exception:
            if model_key!='mobilevit_xxs': raise
            self.backbone=timm.create_model('hf_hub:timm/mobilevit_xxs.cvnets_in1k',pretrained=pretrained,num_classes=0,global_pool='avg')
        d=self.backbone.num_features; self.feature_dim=d
        self.eye_head=nn.Sequential(nn.Dropout(dropout),nn.Linear(d,2))
        self.yawn_head=nn.Sequential(nn.Dropout(dropout),nn.Linear(d,2))
        self.pose_head=nn.Sequential(nn.Dropout(dropout),nn.Linear(d,3),nn.Tanh())

    def features(self,x): return self.backbone(x)
    def forward(self,x,task=None):
        f=self.features(x)
        if task=='eye': return self.eye_head(f)
        if task=='yawn': return self.yawn_head(f)
        if task=='pose': return self.pose_head(f)
        return {'eye':self.eye_head(f),'yawn':self.yawn_head(f),'pose':self.pose_head(f)}

    def forward_multitask(self,eye_x,face_x):
        # 실시간: one model instance, eye ROI + face ROI
        ef=self.features(eye_x); ff=self.features(face_x)
        return self.eye_head(ef), self.yawn_head(ff), self.pose_head(ff)

for k in MODELS_TO_RUN:
    m=MultiTaskDrowsinessModel(k,pretrained=True)
    print(k,'feature',m.feature_dim,'params',f'{sum(p.numel() for p in m.parameters()):,}')
    del m; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()


model.safetensors: reconstructing file:   0%|          |  0.00B / 14.2MB            

model.safetensors: downloading bytes:           |  0.00B            

mobilenetv2 feature 1280 params 2,232,839


model.safetensors: reconstructing file:   0%|          |  0.00B / 46.8MB            

model.safetensors: downloading bytes:           |  0.00B            

resnet18 feature 512 params 11,180,103


model.safetensors: reconstructing file:   0%|          |  0.00B / 5.14MB            

model.safetensors: downloading bytes:           |  0.00B            

mobilevit_xxs feature 320 params 953,271


In [9]:
# 8. Loss / train / evaluate
eye_loss_fn=nn.CrossEntropyLoss(); yawn_loss_fn=nn.CrossEntropyLoss(); pose_loss_fn=nn.SmoothL1Loss(beta=0.1)

def steps_per_task():
    n=min(len(loaders[t]['train']) for t in ['eye','yawn','pose'])
    return n if STEPS_PER_TASK_PER_EPOCH is None else min(n,STEPS_PER_TASK_PER_EPOCH)

def train_epoch(model,opt):
    model.train(); n=steps_per_task(); its={t:iter(loaders[t]['train']) for t in ['eye','yawn','pose']}; hist={t:[] for t in its}
    for _ in range(n):
        order=['eye','yawn','pose']; random.shuffle(order)
        for t in order:
            x,y=next(its[t]); x=x.to(DEVICE,non_blocking=True); y=y.to(DEVICE,non_blocking=True)
            opt.zero_grad(set_to_none=True); out=model(x,task=t)
            if t=='eye': loss=LAMBDA_EYE*eye_loss_fn(out,y)
            elif t=='yawn': loss=LAMBDA_YAWN*yawn_loss_fn(out,y)
            else: loss=LAMBDA_POSE*pose_loss_fn(out,y)
            loss.backward(); opt.step(); hist[t].append(loss.item())
    return {t:float(np.mean(v)) for t,v in hist.items()}

@torch.inference_mode()
def eval_cls(model,task,split):
    model.eval(); fn=eye_loss_fn if task=='eye' else yawn_loss_fn
    total=0.; count=0; yt=[]; yp=[]
    for x,y in loaders[task][split]:
        x=x.to(DEVICE); y=y.to(DEVICE); z=model(x,task=task); loss=fn(z,y)
        total+=loss.item()*len(y); count+=len(y); p=z.argmax(1)
        yt+=y.cpu().tolist(); yp+=p.cpu().tolist()
    acc=accuracy_score(yt,yp); pr,re,f1,_=precision_recall_fscore_support(yt,yp,average='macro',zero_division=0)
    return {'loss':total/count,'accuracy':acc,'precision':pr,'recall':re,'f1':f1}

@torch.inference_mode()
def eval_pose(model,split):
    model.eval(); total=0.; count=0; ys=[]; ps=[]
    for x,y in loaders['pose'][split]:
        x=x.to(DEVICE); y=y.to(DEVICE); p=model(x,task='pose'); loss=pose_loss_fn(p,y)
        total+=loss.item()*len(y); count+=len(y); ys.append((y*POSE_SCALE).cpu()); ps.append((p*POSE_SCALE).cpu())
    y=torch.cat(ys).numpy(); p=torch.cat(ps).numpy(); mae=np.mean(np.abs(y-p),axis=0)
    return {'loss':total/count,'pitch_mae':float(mae[0]),'yaw_mae':float(mae[1]),'roll_mae':float(mae[2]),'mean_mae':float(mae.mean())}

def eval_all(model,split):
    e=eval_cls(model,'eye',split); y=eval_cls(model,'yawn',split); p=eval_pose(model,split)
    total=LAMBDA_EYE*e['loss']+LAMBDA_YAWN*y['loss']+LAMBDA_POSE*p['loss']
    return {'eye':e,'yawn':y,'pose':p,'total_loss':total}


In [10]:
# 9. Backbone 하나 학습 -> .pth 하나 저장
def train_model(model_key):
    print()
    print('=' * 80)
    print(model_key, 'MULTI-TASK')
    print('=' * 80)
    set_seed(SEED)
    model=MultiTaskDrowsinessModel(model_key,pretrained=True).to(DEVICE)
    opt=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=WEIGHT_DECAY)
    sch=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=EPOCHS)
    ckpt=RESULT_DIR/f'{model_key}_multitask_best.pth'
    best=float('inf'); best_epoch=0; stale=0; rows=[]
    print('steps/task/epoch:',steps_per_task())
    for ep in range(1,EPOCHS+1):
        tr=train_epoch(model,opt); val=eval_all(model,'val'); sch.step()
        row={'epoch':ep,'train_eye_loss':tr['eye'],'train_yawn_loss':tr['yawn'],'train_pose_loss':tr['pose'],
             'val_total_loss':val['total_loss'],'val_eye_f1':val['eye']['f1'],'val_yawn_f1':val['yawn']['f1'],
             'val_pitch_mae':val['pose']['pitch_mae'],'val_yaw_mae':val['pose']['yaw_mae'],'val_roll_mae':val['pose']['roll_mae'],'val_pose_mean_mae':val['pose']['mean_mae']}
        rows.append(row)
        print(f"[{ep:02d}/{EPOCHS}] total={val['total_loss']:.4f} | EyeF1={val['eye']['f1']:.4f} | YawnF1={val['yawn']['f1']:.4f} | PoseMAE={val['pose']['mean_mae']:.2f}°")
        if val['total_loss']<best:
            best=val['total_loss']; best_epoch=ep; stale=0
            torch.save({
                'model_type':'multitask_drowsiness','model_key':model_key,'timm_model_name':MODEL_MAP[model_key],
                'model_state_dict':model.state_dict(),'feature_dim':model.feature_dim,
                'tasks':['eye','yawn','pose'],'eye_classes':['OPEN','CLOSED'],'yawn_classes':['NO_YAWN','YAWN'],
                'pose_outputs':['pitch','yaw','roll'],'pose_scale':POSE_SCALE,'pose_unit':'degree',
                'input_size':INPUT_SIZE,'imagenet_mean':MEAN,'imagenet_std':STD,
                'best_epoch':best_epoch,'best_val_loss':best,
                'inference_input':{'eye':'eye_roi','yawn':'face_roi','pose':'face_roi'},
            },ckpt)
            print(' -> saved',ckpt.name)
        else:
            stale+=1
        if stale>=EARLY_STOP_PATIENCE:
            print('Early stopping'); break
    h=pd.DataFrame(rows); h.to_csv(RESULT_DIR/f'{model_key}_multitask_history.csv',index=False)
    return ckpt,h,sum(p.numel() for p in model.parameters())

In [11]:
# 10. 선택한 3 backbone 학습
outputs={}
for k in MODELS_TO_RUN:
    ckpt,h,params=train_model(k)
    outputs[k]={'checkpoint':ckpt,'history':h,'parameters':params}
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()



mobilenetv2 MULTI-TASK
steps/task/epoch: 112
[01/20] total=0.5901 | EyeF1=0.8404 | YawnF1=0.9674 | PoseMAE=15.68°
 -> saved mobilenetv2_multitask_best.pth
[02/20] total=0.4818 | EyeF1=0.8717 | YawnF1=0.9844 | PoseMAE=13.82°
 -> saved mobilenetv2_multitask_best.pth
[03/20] total=0.3392 | EyeF1=0.9232 | YawnF1=0.9896 | PoseMAE=12.20°
 -> saved mobilenetv2_multitask_best.pth
[04/20] total=0.2621 | EyeF1=0.9459 | YawnF1=0.9909 | PoseMAE=9.76°
 -> saved mobilenetv2_multitask_best.pth
[05/20] total=0.2938 | EyeF1=0.9315 | YawnF1=0.9857 | PoseMAE=8.90°
[06/20] total=0.3221 | EyeF1=0.9191 | YawnF1=0.9896 | PoseMAE=8.23°
[07/20] total=0.2169 | EyeF1=0.9530 | YawnF1=0.9948 | PoseMAE=7.91°
 -> saved mobilenetv2_multitask_best.pth
[08/20] total=0.2412 | EyeF1=0.9453 | YawnF1=0.9857 | PoseMAE=7.24°
[09/20] total=0.2158 | EyeF1=0.9485 | YawnF1=0.9870 | PoseMAE=7.49°
 -> saved mobilenetv2_multitask_best.pth
[10/20] total=0.2226 | EyeF1=0.9477 | YawnF1=0.9896 | PoseMAE=7.20°
[11/20] total=0.2054 | Ey

In [12]:
# 11. Best checkpoint Test 평가 + 비교 표
def load_checkpoint(path):
    c=torch.load(path,map_location=DEVICE); m=MultiTaskDrowsinessModel(c['model_key'],pretrained=False)
    m.load_state_dict(c['model_state_dict']); return m.to(DEVICE).eval(),c

rows=[]
for k,info in outputs.items():
    m,c=load_checkpoint(info['checkpoint']); t=eval_all(m,'test')
    rows.append({
      'model':k,'parameters':info['parameters'],'checkpoint_mb':info['checkpoint'].stat().st_size/1024/1024,
      'eye_accuracy':t['eye']['accuracy'],'eye_f1':t['eye']['f1'],
      'yawn_accuracy':t['yawn']['accuracy'],'yawn_f1':t['yawn']['f1'],
      'pitch_mae':t['pose']['pitch_mae'],'yaw_mae':t['pose']['yaw_mae'],'roll_mae':t['pose']['roll_mae'],'pose_mean_mae':t['pose']['mean_mae'],
      'best_epoch':c['best_epoch']})
    del m; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
summary=pd.DataFrame(rows); summary.to_csv(RESULT_DIR/'multitask_test_summary.csv',index=False)
display(summary)


,model,parameters,checkpoint_mb,eye_accuracy,eye_f1,yawn_accuracy,yawn_f1,pitch_mae,yaw_mae,roll_mae,pose_mean_mae,best_epoch
0,mobilenetv2,2232839,8.765639,0.883649,0.882738,0.983073,0.983063,10.280458,9.784409,10.166587,10.077151,15
1,resnet18,11180103,42.731387,0.882718,0.881781,0.980469,0.980463,9.709899,14.515831,11.942370,12.056034,15
2,mobilevit_xxs,953271,3.783232,0.915296,0.915025,0.977865,0.977854,9.769371,11.506921,11.870812,11.049034,10


In [13]:
# 12. 하나의 .pth에서 3 Task 출력이 가능한지 shape 확인
k=list(outputs.keys())[0]
model,ckpt=load_checkpoint(outputs[k]['checkpoint'])
eye_x=next(iter(loaders['eye']['test']))[0][:2].to(DEVICE)
face_x=next(iter(loaders['yawn']['test']))[0][:2].to(DEVICE)
with torch.inference_mode():
    eye_logits,yawn_logits,pose_norm=model.forward_multitask(eye_x,face_x)
    p_closed=torch.softmax(eye_logits,1)[:,1]
    p_yawn=torch.softmax(yawn_logits,1)[:,1]
    pose_deg=pose_norm*POSE_SCALE
print('checkpoint:',outputs[k]['checkpoint'])
print('Eye logits :',eye_logits.shape,' P(closed)=',p_closed.cpu().numpy())
print('Yawn logits:',yawn_logits.shape,' P(yawn)=',p_yawn.cpu().numpy())
print('Pose       :',pose_deg.shape)
print('degree=')
print(pose_deg.cpu().numpy())

checkpoint: /content/drowsiness_multitask_results/mobilenetv2_multitask_best.pth
Eye logits : torch.Size([2, 2])  P(closed)= [0.99334055 0.9963522 ]
Yawn logits: torch.Size([2, 2])  P(yawn)= [9.9902844e-01 5.9212302e-04]
Pose       : torch.Size([2, 3])
degree=
[[-10.679857    16.46031    -18.182077  ]
 [ -7.345617   -39.887108    -0.42099127]]


## 저장 결과

```text
drowsiness_multitask_results/
├── mobilenetv2_multitask_best.pth
├── resnet18_multitask_best.pth
├── mobilevit_xxs_multitask_best.pth
├── *_multitask_history.csv
└── multitask_test_summary.csv
```

각 `.pth` **하나**가 다음 세 출력을 모두 가집니다.

```text
Eye  : OPEN / CLOSED
Yawn : NO_YAWN / YAWN
Pose : Pitch / Yaw / Roll
```

실시간에서는 ROI detector로 `Eye ROI`와 `Face ROI`만 얻은 뒤 동일 모델의 `forward_multitask(eye_roi, face_roi)`를 호출하면 됩니다.


In [14]:
# 13. 결과 ZIP
shutil.make_archive('/content/drowsiness_multitask_results','zip',RESULT_DIR)
print('Created: /content/drowsiness_multitask_results.zip')
for p in sorted(RESULT_DIR.glob('*')):
    print(p.name, f'{p.stat().st_size/1024/1024:.2f} MB')


Created: /content/drowsiness_multitask_results.zip
mobilenetv2_multitask_best.pth 8.77 MB
mobilenetv2_multitask_history.csv 0.00 MB
mobilevit_xxs_multitask_best.pth 3.78 MB
mobilevit_xxs_multitask_history.csv 0.00 MB
multitask_test_summary.csv 0.00 MB
resnet18_multitask_best.pth 42.73 MB
resnet18_multitask_history.csv 0.00 MB
